## 0 · Setup Path & Imports

In [1]:
import sys
from pathlib import Path

# Add the project root so `app` is importable
PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"✅ Project root: {PROJECT_ROOT}")

# --- Load .env so real API keys are available ---
from dotenv import load_dotenv
load_dotenv(Path(PROJECT_ROOT) / ".env")

# --- Standard library ---
import os
import asyncio
import tempfile
from datetime import datetime, timedelta
from unittest.mock import AsyncMock, MagicMock, patch

# --- App imports ---
from app.config import Settings
from app.db.models import (
    ParsedTransaction,
    Transaction,
    TransactionSummary,
    TransactionType,
    User,
    Workspace,
)
from app.db.repository import Repository
from app.bot.formatting import (
    format_transaction_confirmation,
    format_transaction_list,
    format_transaction_row,
    format_summary,
)
from app.llm.parser import parse_transaction

print("✅ All imports successful")
print(f"🔑 OpenAI key loaded: {'yes' if os.getenv('OPENAI_API_KEY') else 'NO — check .env!'}")

✅ Project root: /Users/analytics/Desktop/tg_finans


/Users/analytics/Desktop/tg_finans/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ All imports successful
🔑 OpenAI key loaded: yes


## 1 · Configuration

Create a `Settings` instance with test values and validate it.

In [2]:
# --- Use real keys from .env ---
settings = Settings()  # reads from environment
errors = settings.validate()

if errors:
    print(f"❌ Settings invalid — {len(errors)} errors:")
    for e in errors:
        print(f"   • {e}")
    raise RuntimeError("Fix .env before continuing")

print(f"✅ Settings valid")
print(f"   Model:    {settings.openai_model}")
print(f"   DB path:  {settings.database_path}")
print(f"   Server:   {settings.host}:{settings.port}")
print(f"   API key:  {settings.openai_api_key[:8]}...{settings.openai_api_key[-4:]}")

✅ Settings valid
   Model:    gpt-4o-mini
   DB path:  data/finance.db
   Server:   0.0.0.0:8000
   API key:  sk-proj-...pTUA


## 2 · Initialize Database & Create Workspace

Open a temp SQLite database, run schema migrations, and create a shared workspace.

In [3]:
# Create a temp DB file for this notebook session
_tmp = tempfile.NamedTemporaryFile(suffix=".db", delete=False)
DB_PATH = _tmp.name
_tmp.close()

repo = Repository(DB_PATH)
await repo.connect()
print(f"✅ Database connected: {DB_PATH}")

# Create a workspace
workspace = await repo.create_workspace()
print(f"✅ Workspace created")
print(f"   🔑 Hash:       {workspace.id_hash}")
print(f"   📅 Created at: {workspace.created_at.isoformat()}")

# Retrieve it back
found = await repo.get_workspace(workspace.id_hash)
assert found is not None
assert found.id_hash == workspace.id_hash
print(f"✅ Workspace retrieved successfully")

# Non-existent workspace
missing = await repo.get_workspace("doesnotexist")
assert missing is None
print(f"✅ Non-existent workspace returns None — correct")

✅ Database connected: /var/folders/8_/8kvsb17s1zjc3bhv4jnhqxv00000gn/T/tmpbgxb9pmn.db
✅ Workspace created
   🔑 Hash:       e3c4f499df05
   📅 Created at: 2026-02-17T21:00:33.309655
✅ Workspace retrieved successfully
✅ Non-existent workspace returns None — correct


## 3 · Create User & Link to Workspace

Register a new Telegram user, assign a name and currency, link to the workspace.

In [4]:
# Simulate /start → "Create new database" → enter name → enter currency
user = User(
    telegram_user_id=100500,
    name="Alice",
    default_currency="ILS",
    workspace_id_hash=workspace.id_hash,
)
await repo.upsert_user(user)

fetched = await repo.get_user(100500)
assert fetched is not None
assert fetched.name == "Alice"
assert fetched.workspace_id_hash == workspace.id_hash

print(f"✅ User created & linked to workspace")
print(f"   👤 Name:      {fetched.name}")
print(f"   🆔 TG ID:     {fetched.telegram_user_id}")
print(f"   💱 Currency:  {fetched.default_currency}")
print(f"   🔗 Workspace: {fetched.workspace_id_hash}")

✅ User created & linked to workspace
   👤 Name:      Alice
   🆔 TG ID:     100500
   💱 Currency:  ILS
   🔗 Workspace: e3c4f499df05


## 4 · Write 10 Transactions via Natural Language + Real LLM

Each message is exactly what a user would type in Telegram.  
We call the **real** `parse_transaction()` → the real OpenAI API (gpt-4o-mini) parses it → we save the result.  
This is the exact same pipeline the bot uses in production.

In [5]:
now = datetime(2026, 2, 17, 12, 0, 0)

# 10 natural-language messages exactly as a user would type them in Telegram
USER_MESSAGES = [
    "Coffee 12.5",
    "Got salary 15000",
    "Uber to office 38.20",
    "Supermarket groceries 230",
    "Netflix subscription 55",
    "Freelance project 2500",
    "Electricity bill 380",
    "Restaurant dinner yesterday 185",
    "Gym membership 199",
    "Sold old laptop 800",
]

saved_ids = []
for raw_message in USER_MESSAGES:
    # ── Call the REAL LLM to parse the natural-language message ──
    parsed = await parse_transaction(
        raw_message,
        api_key=settings.openai_api_key,
        model=settings.openai_model,
        default_currency=user.default_currency,
    )

    # Resolve the datetime from the LLM output
    try:
        ts = datetime.fromisoformat(parsed.datetime_str)
    except (ValueError, TypeError):
        ts = now

    # Save to DB (same logic as the bot handler)
    tx = Transaction(
        workspace_id_hash=workspace.id_hash,
        user_id=user.telegram_user_id,
        type=parsed.type,
        category=parsed.category,
        amount=parsed.amount,
        currency=parsed.currency,
        timestamp=ts,
        description=parsed.description,
        raw_text=raw_message,
    )
    tx_id = await repo.add_transaction(tx)
    saved_ids.append(tx_id)
    tx.id = tx_id

    # Print what the LLM extracted + the confirmation
    print(f'💬 User typed: "{raw_message}"')
    print(f'   🤖 LLM parsed → {parsed.type.value} | {parsed.category} | '
          f'{parsed.amount} {parsed.currency} | {parsed.datetime_str}')
    print(format_transaction_confirmation(tx))
    print()

print(f"✅ {len(saved_ids)} transactions parsed by real LLM & saved  (IDs: {saved_ids})")

/Users/analytics/Desktop/tg_finans/.venv/lib/python3.9/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ParsedTransaction(type=<T...ption='Coffee purchase'), input_type=ParsedTransaction])
  return self.__pydantic_serializer__.to_python(


💬 User typed: "Coffee 12.5"
   🤖 LLM parsed → expense | food | 12.5 ILS | 2026-02-17T21:00:39.400422
💸 *Recorded*
  • Type: `Expense`
  • Category: `food`
  • Amount: `12.50 ILS`
  • Date: `2026-02-17 21:00`
  • Description: _Coffee purchase_



/Users/analytics/Desktop/tg_finans/.venv/lib/python3.9/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ParsedTransaction(type=<T...ption='Received salary'), input_type=ParsedTransaction])
  return self.__pydantic_serializer__.to_python(


💬 User typed: "Got salary 15000"
   🤖 LLM parsed → income | salary | 15000.0 ILS | 2026-02-17T21:00:43.560093
💰 *Recorded*
  • Type: `Income`
  • Category: `salary`
  • Amount: `15,000.00 ILS`
  • Date: `2026-02-17 21:00`
  • Description: _Received salary_



/Users/analytics/Desktop/tg_finans/.venv/lib/python3.9/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ParsedTransaction(type=<T...iption='Uber to office'), input_type=ParsedTransaction])
  return self.__pydantic_serializer__.to_python(


...

/Users/analytics/Desktop/tg_finans/.venv/lib/python3.9/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ParsedTransaction(type=<T...ion='Restaurant dinner'), input_type=ParsedTransaction])
  return self.__pydantic_serializer__.to_python(


💬 User typed: "Restaurant dinner yesterday 185"
   🤖 LLM parsed → expense | food | 185.0 ILS | 2026-02-16T21:00:56.502723
💸 *Recorded*
  • Type: `Expense`
  • Category: `food`
  • Amount: `185.00 ILS`
  • Date: `2026-02-16 21:00`
  • Description: _Restaurant dinner_



/Users/analytics/Desktop/tg_finans/.venv/lib/python3.9/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ParsedTransaction(type=<T...iption='Gym membership'), input_type=ParsedTransaction])
  return self.__pydantic_serializer__.to_python(


💬 User typed: "Gym membership 199"
   🤖 LLM parsed → expense | other | 199.0 ILS | 2026-02-17T21:00:57Z
💸 *Recorded*
  • Type: `Expense`
  • Category: `other`
  • Amount: `199.00 ILS`
  • Date: `2026-02-17 12:00`
  • Description: _Gym membership_

💬 User typed: "Sold old laptop 800"
   🤖 LLM parsed → income | other | 800.0 ILS | 2026-02-17T21:00:59Z
💰 *Recorded*
  • Type: `Income`
  • Category: `other`
  • Amount: `800.00 ILS`
  • Date: `2026-02-17 12:00`
  • Description: _Sold old laptop_

✅ 10 transactions parsed by real LLM & saved  (IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10])


/Users/analytics/Desktop/tg_finans/.venv/lib/python3.9/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ParsedTransaction(type=<T...ption='Sold old laptop'), input_type=ParsedTransaction])
  return self.__pydantic_serializer__.to_python(


## 5 · `/transactions` — List & Filter

Simulate the four variants of the `/transactions` command:
1. **All** — no arguments
2. **Single day** — `/transactions 2026-02-17`
3. **Range to now** — `/transactions 2026-02-14 now`
4. **Date range** — `/transactions 2026-02-10 2026-02-15`

In [6]:
# ── 5a) /transactions  (all) ──────────────────────────────────────────────
all_txs = await repo.get_transactions(workspace.id_hash)
print("━" * 60)
print("📋  /transactions   (all)")
print("━" * 60)
print(format_transaction_list(all_txs))

summary_all = await repo.summarize_transactions(
    workspace.id_hash, currency=user.default_currency,
)
print(format_summary(summary_all))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋  /transactions   (all)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 *Transactions* (10 total)

💸 `2026-02-17` | groceries | 230.00 ILS — _Supermarket groceries_ (user 100500)
💰 `2026-02-17` | salary | 15,000.00 ILS — _Received salary_ (user 100500)
💸 `2026-02-17` | food | 12.50 ILS — _Coffee purchase_ (user 100500)
💸 `2026-02-17` | transport | 38.20 ILS — _Uber to office_ (user 100500)
💸 `2026-02-17` | entertainment | 55.00 ILS — _Netflix subscription_ (user 100500)
💰 `2026-02-17` | other | 2,500.00 ILS — _Freelance project income_ (user 100500)
💸 `2026-02-17` | utilities | 380.00 ILS — _Electricity bill_ (user 100500)
💸 `2026-02-17` | other | 199.00 ILS — _Gym membership_ (user 100500)
💰 `2026-02-17` | other | 800.00 ILS — _Sold old laptop_ (user 100500)
💸 `2026-02-16` | food | 185.00 ILS — _Restaurant dinner_ (user 100500)

📊 *Summary* (10 transactions)
  💰 Income:   `18,300.00 ILS`
  💸 Expenses: `1,099.

In [7]:
# ── 5b) /transactions 2026-02-17  (single day) ───────────────────────────
day_start = datetime(2026, 2, 17, 0, 0, 0)
day_end   = datetime(2026, 2, 17, 23, 59, 59)

day_txs = await repo.get_transactions(
    workspace.id_hash, date_from=day_start, date_to=day_end,
)
day_summary = await repo.summarize_transactions(
    workspace.id_hash, date_from=day_start, date_to=day_end,
    currency=user.default_currency,
)

print("━" * 60)
print("📋  /transactions 2026-02-17")
print("━" * 60)
print(format_transaction_list(day_txs))
print(format_summary(day_summary))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋  /transactions 2026-02-17
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 *Transactions* (9 total)

💸 `2026-02-17` | groceries | 230.00 ILS — _Supermarket groceries_ (user 100500)
💰 `2026-02-17` | salary | 15,000.00 ILS — _Received salary_ (user 100500)
💸 `2026-02-17` | food | 12.50 ILS — _Coffee purchase_ (user 100500)
💸 `2026-02-17` | transport | 38.20 ILS — _Uber to office_ (user 100500)
💸 `2026-02-17` | entertainment | 55.00 ILS — _Netflix subscription_ (user 100500)
💰 `2026-02-17` | other | 2,500.00 ILS — _Freelance project income_ (user 100500)
💸 `2026-02-17` | utilities | 380.00 ILS — _Electricity bill_ (user 100500)
💸 `2026-02-17` | other | 199.00 ILS — _Gym membership_ (user 100500)
💰 `2026-02-17` | other | 800.00 ILS — _Sold old laptop_ (user 100500)

📊 *Summary* (9 transactions)
  💰 Income:   `18,300.00 ILS`
  💸 Expenses: `914.70 ILS`
  📈 Net:      `17,385.30 ILS`


In [8]:
# ── 5c) /transactions 2026-02-14 now  (range to now) ─────────────────────
range_start = datetime(2026, 2, 14, 0, 0, 0)

range_txs = await repo.get_transactions(
    workspace.id_hash, date_from=range_start, date_to=now,
)
range_summary = await repo.summarize_transactions(
    workspace.id_hash, date_from=range_start, date_to=now,
    currency=user.default_currency,
)

print("━" * 60)
print("📋  /transactions 2026-02-14 now")
print("━" * 60)
print(format_transaction_list(range_txs))
print(format_summary(range_summary))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋  /transactions 2026-02-14 now
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 *Transactions* (7 total)

💸 `2026-02-17` | transport | 38.20 ILS — _Uber to office_ (user 100500)
💸 `2026-02-17` | entertainment | 55.00 ILS — _Netflix subscription_ (user 100500)
💰 `2026-02-17` | other | 2,500.00 ILS — _Freelance project income_ (user 100500)
💸 `2026-02-17` | utilities | 380.00 ILS — _Electricity bill_ (user 100500)
💸 `2026-02-17` | other | 199.00 ILS — _Gym membership_ (user 100500)
💰 `2026-02-17` | other | 800.00 ILS — _Sold old laptop_ (user 100500)
💸 `2026-02-16` | food | 185.00 ILS — _Restaurant dinner_ (user 100500)

📊 *Summary* (7 transactions)
  💰 Income:   `3,300.00 ILS`
  💸 Expenses: `857.20 ILS`
  📈 Net:      `2,442.80 ILS`


In [9]:
# ── 5d) /transactions 2026-02-10 2026-02-15  (custom range) ──────────────
d1 = datetime(2026, 2, 10, 0, 0, 0)
d2 = datetime(2026, 2, 15, 23, 59, 59)

range2_txs = await repo.get_transactions(
    workspace.id_hash, date_from=d1, date_to=d2,
)
range2_summary = await repo.summarize_transactions(
    workspace.id_hash, date_from=d1, date_to=d2,
    currency=user.default_currency,
)

print("━" * 60)
print("📋  /transactions 2026-02-10 2026-02-15")
print("━" * 60)
print(format_transaction_list(range2_txs))
print(format_summary(range2_summary))

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋  /transactions 2026-02-10 2026-02-15
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📭 No transactions found.

📊 *Summary* (0 transactions)
  💰 Income:   `0.00 ILS`
  💸 Expenses: `0.00 ILS`
  📈 Net:      `0.00 ILS`


## 6 · `/question` — Real LLM Q&A

Ask natural-language questions about your finances using the **real** OpenAI API.  
The LLM generates SQL, the tool executes it against your database, and the LLM summarises the results.

In [10]:
from app.llm.qa import ask_question

QUESTIONS = [
    "How much did I spend in total?",
    "What are my top 5 expense categories?",
    "What is my net income vs expenses?",
    "How much did I spend on food?",
    "What was my biggest single expense?",
]

for q in QUESTIONS:
    print("━" * 60)
    print(f"🤖  /question {q}")
    print("━" * 60)

    answer = await ask_question(
        q,
        api_key=settings.openai_api_key,
        model=settings.openai_model,
        repo=repo,
        workspace_id=workspace.id_hash,
    )

    print(f"\n{answer}\n")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖  /question How much did I spend in total?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

You have spent a total of $1,099.70.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖  /question What are my top 5 expense categories?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Your top 5 expense categories are:

1. **Utilities**: $380.00
2. **Groceries**: $230.00
3. **Other**: $199.00
4. **Food**: $197.50
5. **Entertainment**: $55.00

These categories represent where you've spent the most money recently.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖  /question What is my net income vs expenses?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Your total income is $18,300.00, while your total expenses amount to $1,099.70. This means you have a net income of $17,200.30.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖  /question How much did I

## 7 · Bonus: Raw SQL queries (what the tool actually executes)

Run the SQL that the Q&A agent would generate — directly against the real database.

In [ ]:
# ── Total spent ──
rows = await repo.execute_readonly_sql(
    "SELECT SUM(amount) as total_spent FROM transactions "
    "WHERE workspace_id_hash = ? AND type = 'expense'",
    (workspace.id_hash,),
)
print(f"💸 Total spent: {rows[0]['total_spent']:,.2f} ILS")

# ── Top categories ──
rows = await repo.execute_readonly_sql(
    "SELECT category, SUM(amount) as total, COUNT(*) as cnt "
    "FROM transactions WHERE workspace_id_hash = ? AND type = 'expense' "
    "GROUP BY category ORDER BY total DESC",
    (workspace.id_hash,),
)
print(f"\n📊 Expense breakdown:")
for r in rows:
    print(f"   {r['category']:15s}  {r['total']:>10,.2f} ILS  ({r['cnt']} tx)")

# ── Income sources ──
rows = await repo.execute_readonly_sql(
    "SELECT category, SUM(amount) as total FROM transactions "
    "WHERE workspace_id_hash = ? AND type = 'income' "
    "GROUP BY category ORDER BY total DESC",
    (workspace.id_hash,),
)
print(f"\n💰 Income sources:")
for r in rows:
    print(f"   {r['category']:15s}  {r['total']:>10,.2f} ILS")

# ── Read-only guard: reject writes ──
print()
for bad_sql in [
    "INSERT INTO transactions VALUES (999,'x',1,'income','hack',0,'USD','','','','')",
    "DELETE FROM transactions WHERE id = 1",
    "DROP TABLE transactions",
]:
    try:
        await repo.execute_readonly_sql(bad_sql)
        print(f"❌ Should have blocked: {bad_sql[:40]}...")
    except PermissionError:
        print(f"🛡️  Blocked: {bad_sql[:50]}...")

## 8 · Cleanup

Close the database connection and remove the temp file.

In [ ]:
import os

await repo.close()
os.unlink(DB_PATH)
print(f"✅ Database closed and temp file removed")
print(f"🏁 All pipeline steps completed successfully!")